# Mooby disruption baseline training

This notebook trains two exploratory text-only baselines from the labeled manual corpus:
1. disruption relevance (`disruption` / `not_disruption`);
2. operational severity (`light` / `moderate` / `severe`) on disruption rows with a severity label.

It follows `Spec.md` §16.1: exact duplicate texts are deduplicated before the split, so exact copies cannot leak across train/test. Because most rows have no publication timestamp and the corpus has no content-group IDs, results are exploratory and do not measure time-based or event-group generalization.

The notebook never prints post text. The source CSV is read-only; generated models, metrics, and split IDs are written under this private folder, which is git-ignored.


## 0. Environment

Select the Mooby Python 3.12 virtual environment as this notebook's kernel. If the packages are missing, run this in a terminal with `.venv` activated:

```bash
python -m pip install scikit-learn jupyter ipykernel
```

Restart the notebook kernel after installation.


In [1]:
from pathlib import Path
import csv
import hashlib
import json
import platform
import sys

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)


Python: 3.12.16
scikit-learn: 1.9.1


## 1. Find and load the labeled CSV

The file path is resolved from the repository root. Text is kept unchanged for model input; label normalization is applied only to label fields.


In [2]:
def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "Spec.md").exists():
            return candidate
    raise FileNotFoundError("Could not find Mooby_Back repository root from the notebook working directory")

REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "evaluation/private/Scrape_Manual/text.csv"
OUTPUT_DIR = DATA_PATH.parent / "baseline_artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with DATA_PATH.open("r", encoding="utf-8-sig", newline="") as f:
    source_rows = list(csv.DictReader(f))

required_columns = {"id_baris", "teks", "gangguan_krl", "tingkat_keparahan"}
missing_columns = required_columns - (set(source_rows[0]) if source_rows else set())
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {sorted(missing_columns)}")

print("Source rows:", len(source_rows))
print("Source file:", DATA_PATH)
print("Post text is intentionally not displayed.")


Source rows: 333
Source file: /Users/nfahrisalim/Public/Mooby_Back/evaluation/private/Scrape_Manual/text.csv
Post text is intentionally not displayed.


## 2. Normalize labels and validate rows

The source values `Ya` / `Tidak` map to `disruption` / `not_disruption`. A dash or blank severity maps to null. The original CSV is not modified.


In [3]:
RELEVANCE_MAP = {
    "ya": "disruption",
    "tidak": "not_disruption",
    "disruption": "disruption",
    "not_disruption": "not_disruption",
    "uncertain": "uncertain",
}
SEVERITY_MAP = {
    "light": "light", "ringan": "light",
    "moderate": "moderate", "sedang": "moderate",
    "severe": "severe", "berat": "severe",
}

def norm_relevance(value):
    return RELEVANCE_MAP.get((value or "").strip().casefold(), "__invalid__")

def norm_severity(value):
    value = (value or "").strip().casefold()
    if value in {"", "-", "null", "none", "na", "n/a"}:
        return None
    return SEVERITY_MAP.get(value, "__invalid__")

records = []
for row in source_rows:
    text = row.get("teks") or ""
    relevance = norm_relevance(row.get("gangguan_krl"))
    severity = norm_severity(row.get("tingkat_keparahan"))
    records.append({
        "record_id": (row.get("id_baris") or "").strip(),
        "raw_text": text,
        "relevance_label": relevance,
        "severity_label": severity,
        "published_at": (row.get("published_at") or "").strip(),
        "source": (row.get("sumber") or "").strip(),
    })

blank_text = sum(not r["raw_text"].strip() for r in records)
blank_id = sum(not r["record_id"] for r in records)
invalid_relevance = sum(r["relevance_label"] == "__invalid__" for r in records)
invalid_severity = sum(r["severity_label"] == "__invalid__" for r in records)
print({
    "blank_text": blank_text, "blank_id": blank_id,
    "invalid_relevance": invalid_relevance, "invalid_severity": invalid_severity,
    "timestamped_rows": sum(bool(r["published_at"]) for r in records),
})
if blank_text or blank_id or invalid_relevance or invalid_severity:
    raise ValueError("Resolve these input validation counts before training; no source text was printed.")


{'blank_text': 0, 'blank_id': 0, 'invalid_relevance': 0, 'invalid_severity': 0, 'timestamped_rows': 36}


## 3. Group exact duplicate text and check label conflicts

Each unique exact text is represented once in the modeling table. If exact copies disagree on relevance or severity, the notebook stops and reports IDs only. Near-duplicate or same-incident grouping is unavailable in this source and is not inferred.


In [4]:
from collections import defaultdict, Counter

by_text = defaultdict(list)
for r in records:
    by_text[r["raw_text"]].append(r)

conflict_ids = []
for copies in by_text.values():
    if len({x["relevance_label"] for x in copies}) > 1 or len({x["severity_label"] for x in copies}) > 1:
        conflict_ids.extend(x["record_id"] for x in copies)

if conflict_ids:
    print("Exact duplicate groups with label conflicts (IDs only):", conflict_ids)
    raise ValueError("Resolve conflicting labels for identical text before training.")

canonical = []
for text, copies in by_text.items():
    canonical.append(copies[0])

print("Rows:", len(records))
print("Unique exact texts:", len(canonical))
print("Exact duplicate rows excluded from modeling:", len(records) - len(canonical))
print("Relevance counts (all rows):", dict(Counter(r["relevance_label"] for r in records)))
print("Severity counts (all rows):", dict(Counter(r["severity_label"] or "unresolved" for r in records)))


Rows: 333
Unique exact texts: 320
Exact duplicate rows excluded from modeling: 13
Relevance counts (all rows): {'disruption': 287, 'not_disruption': 46}
Severity counts (all rows): {'light': 131, 'moderate': 96, 'severe': 60, 'unresolved': 46}


## 4. Train/evaluate a baseline

Model: character TF-IDF n-grams plus logistic regression, as specified in `Spec.md`. The split is stratified and uses canonical unique texts, so exact duplicates are not split across partitions. There is no time/event-group split; treat the test metrics as exploratory.


In [7]:
RANDOM_STATE = 42
TEST_SIZE = 0.25

def make_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 5),
            min_df=1,
            max_features=100_000,
            sublinear_tf=True,
        )),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            random_state=RANDOM_STATE,
        )),
    ])

def fit_one_task(rows, label_key, task_name):
    usable = [r for r in rows if r[label_key] not in (None, "", "uncertain")]
    labels = [r[label_key] for r in usable]
    texts = [r["raw_text"] for r in usable]
    counts = Counter(labels)
    if len(counts) < 2 or min(counts.values()) < 4:
        raise ValueError(f"Not enough examples per class for an exploratory split in {task_name}: {dict(counts)}")
    x_train, x_test, y_train, y_test, train_rows, test_rows = train_test_split(
        texts, labels, usable, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=labels
    )
    model = make_model()
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)
    classes = sorted(set(labels))
    report = classification_report(y_test, predictions, labels=classes, output_dict=True, zero_division=0)
    summary = {
        "task": task_name,
        "rows_used": len(usable),
        "class_counts": dict(counts),
        "train_rows": len(x_train),
        "test_rows": len(x_test),
        "test_class_counts": dict(Counter(y_test)),
        "macro_f1": f1_score(y_test, predictions, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "classification_report": report,
        "confusion_matrix_labels": classes,
        "confusion_matrix": confusion_matrix(y_test, predictions, labels=classes).tolist(),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return model, summary, train_rows, test_rows


## 5. Relevance model

`uncertain` rows are excluded from supervised training.


In [8]:
relevance_rows = [r for r in canonical if r["relevance_label"] in {"disruption", "not_disruption", "uncertain"}]
relevance_model, relevance_metrics, relevance_train, relevance_test = fit_one_task(
    relevance_rows, "relevance_label", "disruption_relevance"
)


{
  "task": "disruption_relevance",
  "rows_used": 320,
  "class_counts": {
    "disruption": 274,
    "not_disruption": 46
  },
  "train_rows": 240,
  "test_rows": 80,
  "test_class_counts": {
    "disruption": 69,
    "not_disruption": 11
  },
  "macro_f1": 0.6870109546165883,
  "balanced_accuracy": 0.6600790513833992,
  "classification_report": {
    "disruption": {
      "precision": 0.9041095890410958,
      "recall": 0.9565217391304348,
      "f1-score": 0.9295774647887324,
      "support": 69.0
    },
    "not_disruption": {
      "precision": 0.5714285714285714,
      "recall": 0.36363636363636365,
      "f1-score": 0.4444444444444444,
      "support": 11.0
    },
    "accuracy": 0.875,
    "macro avg": {
      "precision": 0.7377690802348336,
      "recall": 0.6600790513833992,
      "f1-score": 0.6870109546165883,
      "support": 80.0
    },
    "weighted avg": {
      "precision": 0.8583659491193737,
      "recall": 0.875,
      "f1-score": 0.8628716744913927,
      "suppor

## 6. Severity model

Severity is trained separately on disruption rows with one of the three assigned severity labels. A blank/dash severity is not treated as a fourth class.


In [9]:
severity_rows = [
    r for r in canonical
    if r["relevance_label"] == "disruption" and r["severity_label"] in {"light", "moderate", "severe"}
]
severity_model, severity_metrics, severity_train, severity_test = fit_one_task(
    severity_rows, "severity_label", "operational_severity"
)


{
  "task": "operational_severity",
  "rows_used": 274,
  "class_counts": {
    "light": 127,
    "moderate": 92,
    "severe": 55
  },
  "train_rows": 205,
  "test_rows": 69,
  "test_class_counts": {
    "moderate": 23,
    "light": 32,
    "severe": 14
  },
  "macro_f1": 0.8078852182300459,
  "balanced_accuracy": 0.8103649068322981,
  "classification_report": {
    "light": {
      "precision": 0.8787878787878788,
      "recall": 0.90625,
      "f1-score": 0.8923076923076924,
      "support": 32.0
    },
    "moderate": {
      "precision": 0.8095238095238095,
      "recall": 0.7391304347826086,
      "f1-score": 0.7727272727272727,
      "support": 23.0
    },
    "severe": {
      "precision": 0.7333333333333333,
      "recall": 0.7857142857142857,
      "f1-score": 0.7586206896551724,
      "support": 14.0
    },
    "accuracy": 0.8260869565217391,
    "macro avg": {
      "precision": 0.8072150072150072,
      "recall": 0.8103649068322981,
      "f1-score": 0.8078852182300459,
  

## 7. Inspect split coverage without exposing text

This records IDs and labels only. Compare the class balance between train and test; the split remains an exploratory random split, not a time-based or incident-group evaluation.


In [10]:
def split_rows(rows, partition):
    return [{"record_id": r["record_id"], "partition": partition} for r in rows]

manifest = (
    split_rows(relevance_train, "relevance_train")
    + split_rows(relevance_test, "relevance_test")
    + split_rows(severity_train, "severity_train")
    + split_rows(severity_test, "severity_test")
)
manifest_path = OUTPUT_DIR / "split_manifest.csv"
with manifest_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["record_id", "partition"])
    writer.writeheader()
    writer.writerows(manifest)
print("Saved split manifest:", manifest_path)


Saved split manifest: /Users/nfahrisalim/Public/Mooby_Back/evaluation/private/Scrape_Manual/baseline_artifacts/split_manifest.csv


## 8. Save private exploratory artifacts

Model artifacts can contain learned text features. Keep them in the git-ignored private folder and do not commit or publish them without reviewing the repository's data-handling policy.


In [11]:
import joblib
from datetime import datetime, timezone

source_digest = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
metadata = {
    "experiment": "exploratory_text_baseline_v1",
    "source_file_sha256": source_digest,
    "source_rows": len(records),
    "canonical_exact_texts": len(canonical),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "split_method": "stratified random split after exact-text deduplication",
    "temporal_split": False,
    "content_group_split": False,
    "python_version": platform.python_version(),
    "scikit_learn_version": sklearn.__version__,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "relevance_metrics": relevance_metrics,
    "severity_metrics": severity_metrics,
}

# Refit final artifacts on every eligible canonical row after evaluating the holdout split.
relevance_fit_rows = [r for r in canonical if r["relevance_label"] in {"disruption", "not_disruption"}]
final_relevance_model = make_model()
final_relevance_model.fit(
    [r["raw_text"] for r in relevance_fit_rows],
    [r["relevance_label"] for r in relevance_fit_rows],
)
final_severity_model = make_model()
final_severity_model.fit(
    [r["raw_text"] for r in severity_rows],
    [r["severity_label"] for r in severity_rows],
)

joblib.dump(final_relevance_model, OUTPUT_DIR / "relevance_tfidf_logreg.joblib")
joblib.dump(final_severity_model, OUTPUT_DIR / "severity_tfidf_logreg.joblib")
(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Saved private model artifacts and metrics under:", OUTPUT_DIR)


Saved private model artifacts and metrics under: /Users/nfahrisalim/Public/Mooby_Back/evaluation/private/Scrape_Manual/baseline_artifacts


## 9. Interpretation

- These metrics are exploratory only; the data lacks complete publication-time and content/event-group metadata.
- Inspect per-class precision/recall/F1 and confusion matrices, especially for `not_disruption` and `severe`; accuracy alone is not sufficient.
- Do not interpret model confidence as truth verification. The production backend must retain its verification/publication controls.
- This notebook trains classifiers only. Station, line/path, and event-time extraction remain rule-based as described in `Spec.md`.
